# Notebook 03 — Silver Layer: Limpeza, Transformação e Enriquecimento
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 2 — Medallion Architecture & Storage

---

**Objetivo:** Transformar os dados brutos da camada Bronze em dados confiáveis, limpos e enriquecidos na camada **Silver**, usando **Delta Lake** para garantir transações ACID, _schema enforcement_ e _time travel_.

**Pipeline desta camada:**
```
Bronze (Parquet raw)
    ├── trips/        →  Dedup + limpeza + Haversine + features temporais + joins  →  silver/trips/  (Delta)
    ├── weather/      →  Limpeza + normalização + chave de join horária            →  silver/weather/ (Delta)
    └── stations/     →  Dedup + padronização + zona NYC                           →  silver/stations/ (Delta)
```

**Por que Delta Lake?**
| Recurso | Benefício |
|---|---|
| ACID Transactions | Nenhuma leitura suja durante escritas |
| Schema Enforcement | Rejeita dados fora do schema definido |
| Time Travel | Auditoria e rollback por versão ou timestamp |
| Upsert / Merge | Atualização incremental eficiente |
| Compaction | `OPTIMIZE + ZORDER` para consultas rápidas |


## 0. Setup & Dependências

In [1]:
# Instalar dependências (Delta Lake + PySpark)
!pip install pyspark delta-spark pyarrow pandas requests holidays -q

In [2]:
import os
import sys
import json
import platform
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import holidays

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType,
    IntegerType, FloatType, DateType, LongType, BooleanType
)
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('citibike_silver')

print(f"Python {sys.version}")
print(f"Pandas {pd.__version__}")
print(f"PyArrow {pa.__version__}")

Python 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 11:57:02) [GCC 12.3.0]
Pandas 2.0.3
PyArrow 13.0.0


In [3]:
# ============================================================
# MODO DE EXECUÇÃO — controla volume processado pelo Silver
# ============================================================
# DEV  → filtro temporal 2025-12-01 → 2026-01-31 (~3M linhas, iteração rápida)
# PROD → dataset completo (~23,7M linhas, todo o Bronze)
MODE = 'DEV'

if MODE == 'DEV':
    SILVER_DATE_START = '2025-12-01'
    SILVER_DATE_END   = '2026-02-01'  # exclusive (filtra started_at < END)
elif MODE == 'PROD':
    SILVER_DATE_START = None
    SILVER_DATE_END   = None
else:
    raise ValueError(f"MODE inválido: {MODE}. Use 'DEV' ou 'PROD'.")

print(f'MODE: {MODE}')
if MODE == 'DEV':
    print(f'  Filtro temporal: {SILVER_DATE_START} <= started_at < {SILVER_DATE_END}')
else:
    print('  Sem filtro temporal — Silver cobre todo o Bronze')

# ============================================================
# CONFIGURACAO — Caminhos do Data Lake
# ============================================================

PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR     = PROJECT_ROOT / 'dados'

# Camadas
BRONZE_DIR   = DATA_DIR / 'bronze'
SILVER_DIR   = DATA_DIR / 'silver'

# Bronze — entradas
BRONZE_TRIPS    = BRONZE_DIR / 'trips'
BRONZE_WEATHER  = BRONZE_DIR / 'weather'
BRONZE_HOLIDAYS = BRONZE_DIR / 'holidays'
BRONZE_STATIONS = BRONZE_DIR / 'stations'

# Silver — saídas Delta
SILVER_TRIPS    = SILVER_DIR / 'trips'
SILVER_WEATHER  = SILVER_DIR / 'weather'
SILVER_STATIONS = SILVER_DIR / 'stations'

# Criar diretórios se não existirem
for d in [SILVER_TRIPS, SILVER_WEATHER, SILVER_STATIONS]:
    d.mkdir(parents=True, exist_ok=True)

print("Estrutura Silver:")
for d in [SILVER_TRIPS, SILVER_WEATHER, SILVER_STATIONS]:
    print(f"  {d}")

MODE: DEV
  Filtro temporal: 2025-12-01 <= started_at < 2026-02-01
Estrutura Silver:
  /home/jovyan/work/dados/silver/trips
  /home/jovyan/work/dados/silver/weather
  /home/jovyan/work/dados/silver/stations


In [4]:
# ============================================================
# Inicializar SparkSession com suporte a Delta Lake
# ============================================================

os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

if platform.system() == 'Windows':
    _hadoop_home = os.environ.get('HADOOP_HOME', '')
    if _hadoop_home:
        _hadoop_bin = os.path.join(_hadoop_home, 'bin')
        if _hadoop_bin not in os.environ.get('PATH', ''):
            os.environ['PATH'] = _hadoop_bin + ';' + os.environ.get('PATH', '')

# Parar sessão anterior
_active = SparkSession.getActiveSession()
if _active is not None:
    _active.stop()

# configure_spark_with_delta_pip injeta automaticamente o JAR correto
# e configura as extensões SQL do Delta Lake
spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .appName('CitiBike_Silver')
    .config('spark.driver.memory', os.environ.get('SPARK_DRIVER_MEMORY', '8g'))
    .config('spark.sql.shuffle.partitions', '200')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.warehouse.dir',
            str(DATA_DIR / 'spark-warehouse').replace('\\', '/'))
    .config('spark.local.dir',
            str(DATA_DIR / 'spark-temp').replace('\\', '/'))
    # Delta Lake — extensões obrigatórias para Spark 3.5
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.databricks.delta.retentionDurationCheck.enabled', 'false')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .master('local[*]')
).getOrCreate()

# Injetar hadoop.home.dir no JVM (necessário no Windows)
if platform.system() == 'Windows':
    _hadoop_home = os.environ.get('HADOOP_HOME', '')
    if _hadoop_home:
        spark._jvm.System.setProperty(
            'hadoop.home.dir', _hadoop_home.replace('\\', '/'))
        logger.info(f'hadoop.home.dir configurado: {_hadoop_home}')

spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} + Delta Lake inicializado")
print(f"  Cores: {spark.sparkContext.defaultParallelism}")
print(f"  Driver memory: {spark.conf.get('spark.driver.memory')}")
print(f"  Extensões: {spark.conf.get('spark.sql.extensions')}")

Spark 3.5.0 + Delta Lake inicializado
  Cores: 8
  Driver memory: 4g
  Extensões: io.delta.sql.DeltaSparkSessionExtension


---
## 1. Silver Weather — Limpeza e Normalização

Transformações aplicadas sobre `bronze/weather/`:
- Remoção de registros com temperatura nula
- Parse de `datetime` para `TimestampType`
- Criação de chave de join `datetime_hour` (truncado na hora)
- Filtragem de outliers climáticos
- Escrita como **Delta Lake** particionado por `(year, month)`


In [5]:
# ============================================================
# 1.1 Carregar Bronze Weather
# CORREÇÃO: ler via PyArrow para evitar erro TIMESTAMP(NANOS)
# ============================================================

weather_parquet = BRONZE_WEATHER / 'data.parquet'
if not weather_parquet.exists():
    raise FileNotFoundError(f'Bronze weather não encontrado: {weather_parquet}')

pdf_weather = pq.read_table(str(weather_parquet)).to_pandas()

# Converter timestamps para string antes de passar ao Spark
for col in pdf_weather.columns:
    if hasattr(pdf_weather[col], 'dt'):
        pdf_weather[col] = pdf_weather[col].astype(str)

df_weather_raw = spark.createDataFrame(pdf_weather)
print(f'Bronze Weather: {df_weather_raw.count():,} registros')
df_weather_raw.printSchema()
df_weather_raw.show(3, truncate=False)

Bronze Weather: 5,832 registros
root
 |-- datetime: string (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- snow_depth: double (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- date: string (nullable = true)
 |-- hour: long (nullable = true)
 |-- weather_category: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)

+-------------------+--------------+--------------------+--------------------+-------------+----+--------+----------+------------+--------------+--------------+-----------+----+-----+----------+----

In [6]:
# ============================================================
# 1.2 Limpeza e normalização do Weather
# ============================================================

df_weather = (
    df_weather_raw

    # --- Parse de timestamp (vem como string do Pandas) ---
    .withColumn('datetime',
        F.to_timestamp(F.col('datetime'), 'yyyy-MM-dd HH:mm:ss')
    )

    # --- Remover nulos críticos ---
    .filter(F.col('datetime').isNotNull())
    .filter(F.col('temperature_2m').isNotNull())

    # --- Filtrar outliers: temperatura entre -30 e 50°C (NYC extremos históricos) ---
    .filter(F.col('temperature_2m').between(-30, 50))

    # --- Chave de join horária (usada no join com trips) ---
    .withColumn('datetime_hour', F.date_trunc('hour', F.col('datetime')))

    # --- Features temporais ---
    .withColumn('year',  F.year('datetime'))
    .withColumn('month', F.month('datetime'))
    .withColumn('hour',  F.hour('datetime'))

    # --- Normalizar nulos de precipitação para 0 ---
    .withColumn('precipitation',
        F.coalesce(F.col('precipitation'), F.lit(0.0))
    )

    # --- Flag de chuva (> 0.1 mm/h) ---
    .withColumn('is_raining',
        (F.col('precipitation') > 0.1).cast(BooleanType())
    )

    # --- Timestamp de processamento ---
    .withColumn('silver_ingestion_ts', F.current_timestamp())

    # --- Selecionar colunas com nomes corretos do Bronze gerado pelo NB01 ---
    # CORREÇÃO: weather_code → weathercode, wind_speed_10m → windspeed_10m, etc.
    .select(
        'datetime_hour',
        'datetime',
        'year',
        'month',
        'hour',
        'temperature_2m',
        'precipitation',
        'is_raining',
        F.col('weather_code').alias('weathercode'),
        'weather_category',
        F.col('wind_speed_10m').alias('windspeed_10m'),
        F.col('relative_humidity_2m').alias('relativehumidity_2m'),
        'silver_ingestion_ts'
    )

    # --- Deduplicar por hora (garante unicidade da chave de join) ---
    .dropDuplicates(['datetime_hour'])
)

print(f"Silver Weather: {df_weather.count():,} registros")
df_weather.printSchema()
df_weather.show(3, truncate=False)

Silver Weather: 5,832 registros
root
 |-- datetime_hour: timestamp (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- precipitation: double (nullable = false)
 |-- is_raining: boolean (nullable = false)
 |-- weathercode: long (nullable = true)
 |-- weather_category: string (nullable = true)
 |-- windspeed_10m: double (nullable = true)
 |-- relativehumidity_2m: long (nullable = true)
 |-- silver_ingestion_ts: timestamp (nullable = false)

+-------------------+-------------------+----+-----+----+--------------+-------------+----------+-----------+----------------+-------------+-------------------+--------------------------+
|datetime_hour      |datetime           |year|month|hour|temperature_2m|precipitation|is_raining|weathercode|weather_category|windspeed_10m|relativehumidity_2m|silver_ingestion_ts       |
+--------

In [7]:
# ============================================================
# 1.3 Escrever Silver Weather como Delta Lake
# ============================================================

from delta.tables import DeltaTable

silver_weather_path = str(SILVER_WEATHER).replace('\\', '/')

(
    df_weather
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('year', 'month')
    .save(silver_weather_path)
)

# Verificar
dt_weather = DeltaTable.forPath(spark, silver_weather_path)
history = dt_weather.history(1).select('version', 'timestamp', 'operation', 'operationMetrics')
history.show(truncate=False)

count = spark.read.format('delta').load(silver_weather_path).count()
print(f"Silver Weather gravado: {count:,} registros")
print(f"Path: {silver_weather_path}")

+-------+-----------------------+---------+----------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                |
+-------+-----------------------+---------+----------------------------------------------------------------+
|0      |2026-05-04 15:55:04.122|WRITE    |{numFiles -> 8, numOutputRows -> 5832, numOutputBytes -> 129086}|
+-------+-----------------------+---------+----------------------------------------------------------------+

Silver Weather gravado: 5,832 registros
Path: /home/jovyan/work/dados/silver/weather


---
## 2. Silver Stations — Deduplicação e Enriquecimento

Transformações aplicadas sobre `bronze/stations/`:
- Deduplicação por `station_id`
- Padronização de nomes (strip, lowercase → title case)
- Classificação de zona NYC por bounding boxes geográficas
- Escrita como **Delta Lake** (tabela de referência, sem partição)


In [8]:
# ============================================================
# 2.1 Carregar Bronze Stations (station_info)
# CORREÇÃO: ler via PyArrow para evitar erro TIMESTAMP(NANOS)
# ============================================================

station_parquet = BRONZE_STATIONS / 'station_info' / 'data.parquet'
if not station_parquet.exists():
    raise FileNotFoundError(f'Bronze Stations não encontrado: {station_parquet}')

pdf_stations = pq.read_table(str(station_parquet)).to_pandas()

# Converter timestamps para string antes de passar ao Spark
for col in pdf_stations.columns:
    if hasattr(pdf_stations[col], 'dt'):
        pdf_stations[col] = pdf_stations[col].astype(str)

df_stations_raw = spark.createDataFrame(pdf_stations)
print(f'Bronze Stations: {df_stations_raw.count():,} estações')
df_stations_raw.printSchema()
df_stations_raw.show(5, truncate=False)

Bronze Stations: 2,407 estações
root
 |-- station_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- capacity: long (nullable = true)
 |-- region_id: string (nullable = true)
 |-- short_name: string (nullable = true)
 |-- station_type: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- feed_timestamp: long (nullable = true)

+------------------------------------+-----------------------------+-----------------+-----------------+--------+---------+----------+------------+--------------------------+--------------+
|station_id                          |name                         |lat              |lon              |capacity|region_id|short_name|station_type|ingestion_timestamp       |feed_timestamp|
+------------------------------------+-----------------------------+-----------------+-----------------+--------+---------+----------+------------+--------------------------+

In [9]:
# ============================================================
# 2.2 Limpeza e enriquecimento de Stations
# ============================================================

# Bounding boxes aproximadas das zonas NYC (lat/lng)
# Usadas para classificar cada estação por bairro
@F.udf(StringType())
def classify_nyc_zone(lat, lon):
    if lat is None or lon is None:
        return 'Unknown'
    # Manhattan: lat 40.70-40.88, lon -74.02 a -73.91
    if 40.70 <= lat <= 40.88 and -74.02 <= lon <= -73.91:
        return 'Manhattan'
    # Brooklyn: lat 40.57-40.74, lon -74.04 a -73.83
    elif 40.57 <= lat <= 40.74 and -74.04 <= lon <= -73.83:
        return 'Brooklyn'
    # Queens: lat 40.54-40.80, lon -73.96 a -73.70
    elif 40.54 <= lat <= 40.80 and -73.96 <= lon <= -73.70:
        return 'Queens'
    # Bronx: lat 40.78-40.92, lon -73.93 a -73.75
    elif 40.78 <= lat <= 40.92 and -73.93 <= lon <= -73.75:
        return 'Bronx'
    # Jersey City / Hoboken: lon < -74.02
    elif lon < -74.02:
        return 'Jersey City / Hoboken'
    else:
        return 'Other'

df_stations = (
    df_stations_raw

    # --- Remover estações sem coordenadas ---
    .filter(F.col('lat').isNotNull() & F.col('lon').isNotNull())

    # --- Padronização de nome da estação ---
    .withColumn('station_name_clean',
        F.initcap(F.trim(F.col('name')))
    )

    # --- Capacidade: garantir tipo inteiro ---
    .withColumn('capacity',
        F.col('capacity').cast(IntegerType())
    )

    # --- Classificar zona NYC ---
    .withColumn('nyc_zone', classify_nyc_zone(F.col('lat'), F.col('lon')))

    # --- Deduplicar por station_id (manter registro mais recente) ---
    .dropDuplicates(['station_id'])

    # --- Renomear para schema consistente ---
    .select(
        F.col('short_name').alias('station_id'),
        F.col('station_name_clean').alias('station_name'),
        F.col('lat').cast(DoubleType()).alias('latitude'),
        F.col('lon').cast(DoubleType()).alias('longitude'),
        F.col('capacity'),
        F.col('nyc_zone'),
        F.current_timestamp().alias('silver_ingestion_ts')
    )
)

print(f"Silver Stations: {df_stations.count():,} estações únicas")
print("\nDistribuição por zona:")
df_stations.groupBy('nyc_zone').count().orderBy(F.desc('count')).show()
df_stations.show(5, truncate=False)

Silver Stations: 2,407 estações únicas

Distribuição por zona:
+--------------------+-----+
|            nyc_zone|count|
+--------------------+-----+
|           Manhattan| 1120|
|            Brooklyn|  763|
|               Bronx|  244|
|              Queens|  191|
|Jersey City / Hob...|   89|
+--------------------+-----+

+----------+------------------------+-----------------+------------------+--------+---------+--------------------------+
|station_id|station_name            |latitude         |longitude         |capacity|nyc_zone |silver_ingestion_ts       |
+----------+------------------------+-----------------+------------------+--------+---------+--------------------------+
|6879.04   |28 Ave & 44 St          |40.76408932350688|-73.91065120697021|19      |Manhattan|2026-05-04 15:55:12.629167|
|7830.03   |Prospect Ave & E 151 St |40.814232        |-73.903927        |21      |Bronx    |2026-05-04 15:55:12.629167|
|6225.03   |41 Ave & 67 St          |40.74446         |-73.89764      

In [10]:
# ============================================================
# 2.3 Escrever Silver Stations como Delta Lake
# ============================================================

silver_stations_path = str(SILVER_STATIONS).replace('\\', '/')

(
    df_stations
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    # Sem partição — tabela de referência pequena, leitura broadcast
    .save(silver_stations_path)
)

dt_stations = DeltaTable.forPath(spark, silver_stations_path)
dt_stations.history(1).select('version', 'timestamp', 'operation').show(truncate=False)

count = spark.read.format('delta').load(silver_stations_path).count()
print(f"Silver Stations gravado: {count:,} estações")
print(f"Path: {silver_stations_path}")

+-------+---------------------+---------+
|version|timestamp            |operation|
+-------+---------------------+---------+
|0      |2026-05-04 15:55:16.9|WRITE    |
+-------+---------------------+---------+

Silver Stations gravado: 2,407 estações
Path: /home/jovyan/work/dados/silver/stations


---
## 3. Silver Trips — Pipeline Principal

Pipeline completo de transformação das viagens:

```
Bronze Trips (Parquet)
    │
    ├── 3.1  Deduplicação por ride_id
    ├── 3.2  Filtros de qualidade (coordenadas, duração, tipos)
    ├── 3.3  Features temporais (hour, day_of_week, is_weekend, season)
    ├── 3.4  Distância Haversine (distance_km)
    ├── 3.5  Join com Silver Weather (temperatura, precipitação, categoria)
    ├── 3.6  Join com Holidays (flag is_holiday, nome do feriado)
    ├── 3.7  Join com Silver Stations (zona NYC, capacidade — início e fim)
    └── 3.8  Escrita como Delta Lake particionado por (year, month)
```


In [11]:
# ============================================================
# 3.1 Carregar Bronze Trips
# ============================================================

bronze_trips_path = str(BRONZE_TRIPS).replace('\\', '/')

if not any(BRONZE_TRIPS.rglob('*.parquet')):
    raise FileNotFoundError(
        f"Bronze Trips vazio: {BRONZE_TRIPS}\n"
        "Execute o Notebook 01 primeiro."
    )

# Aplica filtro temporal apenas em MODE=DEV (partition pruning year/month)
df_trips_raw = (
    spark.read
    .format('parquet')
    .load(bronze_trips_path)
)


if MODE == 'DEV':
    # Partition pruning via year/month + reforço por started_at
    df_trips_raw = (
        df_trips_raw
        .filter(
            ((F.col('year') == 2025) & (F.col('month') == 12)) |
            ((F.col('year') == 2026) & (F.col('month') == 1))
        )
        .filter(F.col('started_at') >= F.lit(SILVER_DATE_START).cast('timestamp'))
        .filter(F.col('started_at') <  F.lit(SILVER_DATE_END  ).cast('timestamp'))
    )

total_raw = df_trips_raw.count()
print(f"Bronze Trips carregado: {total_raw:,} registros")
print(f"Schema:")
df_trips_raw.printSchema()
df_trips_raw.show(3, truncate=False)

Bronze Trips carregado: 3,907,608 registros
Schema:
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+----------------+-------------+-----------------------+-----------------------+------------------------+----------------+------------------------+--------------+-----------+------------+----------

In [12]:
# ============================================================
# 3.2 Deduplicação por ride_id
# ============================================================

df_dedup = (
    df_trips_raw
    # Manter apenas 1 registro por ride_id (o mais recente pelo ingestion_timestamp)
    .dropDuplicates(['ride_id'])
)

total_dedup = df_dedup.count()
removed = total_raw - total_dedup
print(f"Antes: {total_raw:,} registros")
print(f"Após dedup: {total_dedup:,} registros")
print(f"Removidos (duplicatas): {removed:,} ({removed/total_raw*100:.2f}%)")

Antes: 3,907,608 registros
Após dedup: 3,907,608 registros
Removidos (duplicatas): 0 (0.00%)


In [13]:
# ============================================================
# 3.3 Filtros de Qualidade
# ============================================================

df_clean = (
    df_dedup

    # --- Parse de timestamps (se ainda como string) ---
    .withColumn('started_at',
        F.to_timestamp(F.col('started_at'))
    )
    .withColumn('ended_at',
        F.to_timestamp(F.col('ended_at'))
    )

    # --- Calcular duração em segundos ---
    .withColumn('trip_duration_sec',
        F.unix_timestamp('ended_at') - F.unix_timestamp('started_at')
    )

    # --- Filtrar registros sem timestamps válidos ---
    .filter(F.col('started_at').isNotNull())
    .filter(F.col('ended_at').isNotNull())

    # --- Filtrar durações inválidas:
    #     < 60s  = provavelmente erro ou viagem cancelada
    #     > 24h  = outlier (86400 segundos) ---
    .filter(F.col('trip_duration_sec').between(60, 86400))

    # --- Filtrar coordenadas inválidas (lat/lng nulos ou fora de NYC área) ---
    .filter(F.col('start_lat').isNotNull() & F.col('start_lng').isNotNull())
    .filter(F.col('end_lat').isNotNull()   & F.col('end_lng').isNotNull())
    .filter(F.col('start_lat').between(40.4, 41.0))
    .filter(F.col('start_lng').between(-74.3, -73.6))
    .filter(F.col('end_lat').between(40.4, 41.0))
    .filter(F.col('end_lng').between(-74.3, -73.6))

    # --- Filtrar tipos válidos ---
    .filter(F.col('rideable_type').isin('classic_bike', 'electric_bike', 'docked_bike'))
    .filter(F.col('member_casual').isin('member', 'casual'))
)

total_clean = df_clean.count()
removed_q = total_dedup - total_clean
print(f"Após dedup: {total_dedup:,} registros")
print(f"Após limpeza: {total_clean:,} registros")
print(f"Removidos (qualidade): {removed_q:,} ({removed_q/total_dedup*100:.2f}%)")
print("\nDistribuição por tipo de bike:")
df_clean.groupBy('rideable_type').count().show()
print("Distribuição member/casual:")
df_clean.groupBy('member_casual').count().show()

Após dedup: 3,907,608 registros
Após limpeza: 3,891,295 registros
Removidos (qualidade): 16,313 (0.42%)

Distribuição por tipo de bike:
+-------------+-------+
|rideable_type|  count|
+-------------+-------+
|electric_bike|2820888|
| classic_bike|1070407|
+-------------+-------+

Distribuição member/casual:
+-------------+-------+
|member_casual|  count|
+-------------+-------+
|       casual| 378748|
|       member|3512547|
+-------------+-------+



In [14]:
# ============================================================
# 3.4 Features Temporais + Distância Haversine
# ============================================================

# Constante do raio da Terra em km
EARTH_RADIUS_KM = F.lit(6371.0)

def haversine_col(lat1, lon1, lat2, lon2):
    """
    Calcula distância Haversine entre dois pontos (em km)
    usando colunas PySpark.
    """
    dlat = F.radians(lat2 - lat1)
    dlon = F.radians(lon2 - lon1)

    a = (
        F.pow(F.sin(dlat / 2), 2) +
        F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) *
        F.pow(F.sin(dlon / 2), 2)
    )
    c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
    return F.round(EARTH_RADIUS_KM * c, 3)

# Mapeamento de estação do ano (hemisfério norte / NYC)
def season_col(month_col):
    return (
        F.when(month_col.isin(12, 1, 2), F.lit('Winter'))
         .when(month_col.isin(3, 4, 5),  F.lit('Spring'))
         .when(month_col.isin(6, 7, 8),  F.lit('Summer'))
         .otherwise(F.lit('Fall'))
    )

df_features = (
    df_clean

    # --- Features de tempo ---
    .withColumn('date',          F.to_date('started_at'))
    .withColumn('hour_of_day',   F.hour('started_at'))
    .withColumn('day_of_week',   F.dayofweek('started_at'))       # 1=Dom, 7=Sáb
    .withColumn('day_name',      F.date_format('started_at', 'EEEE'))
    .withColumn('is_weekend',
        F.col('day_of_week').isin(1, 7).cast(BooleanType())
    )
    .withColumn('season',        season_col(F.month('started_at')))
    .withColumn('year',          F.year('started_at'))
    .withColumn('month',         F.month('started_at'))

    # --- Chave de join horária para o weather ---
    .withColumn('started_at_hour', F.date_trunc('hour', 'started_at'))

    # --- Distância Haversine ---
    .withColumn('distance_km',
        haversine_col(
            F.col('start_lat'), F.col('start_lng'),
            F.col('end_lat'),   F.col('end_lng')
        )
    )

    # --- Velocidade média estimada (km/h) — útil para filtrar outliers futuros ---
    .withColumn('avg_speed_kmh',
        F.round(
            F.col('distance_km') / (F.col('trip_duration_sec') / 3600.0),
            2
        )
    )

    # Filtrar velocidade absurda (> 60 km/h provavelmente erro de GPS)
    .filter(
        F.col('distance_km').isNull() |
        F.col('avg_speed_kmh').isNull() |
        F.col('avg_speed_kmh').between(0, 60)
    )
)

print(f"Após features: {df_features.count():,} registros")
print("\nAmostra de features calculadas:")
df_features.select(
    'ride_id', 'started_at', 'hour_of_day', 'day_name',
    'is_weekend', 'season', 'distance_km', 'trip_duration_sec'
).show(5, truncate=False)

Após features: 3,891,292 registros

Amostra de features calculadas:
+----------------+-----------------------+-----------+--------+----------+------+-----------+-----------------+
|ride_id         |started_at             |hour_of_day|day_name|is_weekend|season|distance_km|trip_duration_sec|
+----------------+-----------------------+-----------+--------+----------+------+-----------+-----------------+
|00002118D4DB6708|2025-12-22 13:17:32.269|13         |Monday  |false     |Winter|1.34       |508              |
|00002E636893DB48|2025-12-02 16:39:41.717|16         |Tuesday |false     |Winter|1.096      |389              |
|000033136BC77C57|2025-12-01 07:24:35.504|7          |Monday  |false     |Winter|0.672      |209              |
|0000541E5009D5CF|2025-12-13 21:45:49.327|21         |Saturday|true      |Winter|0.444      |110              |
|000092438389A01C|2026-01-24 10:16:33.988|10         |Saturday|true      |Winter|0.99       |266              |
+----------------+------------------

In [15]:
# ============================================================
# 3.5 Join com Silver Weather
# ============================================================

silver_weather_path = str(SILVER_WEATHER).replace('\\', '/')

df_weather_silver = (
    spark.read.format('delta').load(silver_weather_path)
    .select(
        'datetime_hour',
        F.col('temperature_2m').alias('weather_temp_c'),
        F.col('precipitation').alias('weather_precip_mm'),
        F.col('is_raining').alias('weather_is_raining'),
        F.col('weathercode').alias('weather_code'),
        F.col('weather_category')
    )
)

# Left join: manter todas as viagens, mesmo sem dado climático
df_with_weather = (
    df_features
    .join(
        F.broadcast(df_weather_silver),   # broadcast pois weather é pequeno
        df_features['started_at_hour'] == df_weather_silver['datetime_hour'],
        how='left'
    )
    .drop('datetime_hour')
)

# Verificar cobertura do join
total = df_with_weather.count()
matched = df_with_weather.filter(F.col('weather_temp_c').isNotNull()).count()
print(f"Total registros: {total:,}")
print(f"Com dados climáticos: {matched:,} ({matched/total*100:.1f}%)")
df_with_weather.select(
    'ride_id', 'started_at', 'weather_temp_c',
    'weather_is_raining', 'weather_category'
).show(5, truncate=False)

Total registros: 3,891,292
Com dados climáticos: 3,891,292 (100.0%)
+----------------+-----------------------+--------------+------------------+----------------+
|ride_id         |started_at             |weather_temp_c|weather_is_raining|weather_category|
+----------------+-----------------------+--------------+------------------+----------------+
|00002118D4DB6708|2025-12-22 13:17:32.269|2.0           |false             |clear           |
|00002E636893DB48|2025-12-02 16:39:41.717|2.8           |true              |rain            |
|000033136BC77C57|2025-12-01 07:24:35.504|2.3           |false             |clear           |
|0000541E5009D5CF|2025-12-13 21:45:49.327|0.9           |true              |snow            |
|000092438389A01C|2026-01-24 10:16:33.988|-11.9         |false             |clear           |
+----------------+-----------------------+--------------+------------------+----------------+
only showing top 5 rows



In [16]:
# ============================================================
# 3.6 Join com Feriados (Bronze Holidays)
# CORREÇÃO: ler via PyArrow para evitar erro TIMESTAMP(NANOS)
# ============================================================

holidays_parquet = BRONZE_HOLIDAYS / 'data.parquet'
pdf_holidays = pq.read_table(str(holidays_parquet)).to_pandas()

# Converter timestamps para string antes de passar ao Spark
for col in pdf_holidays.columns:
    if hasattr(pdf_holidays[col], 'dt'):
        pdf_holidays[col] = pdf_holidays[col].astype(str)

df_holidays = (
    spark.createDataFrame(pdf_holidays)
    .select(
        F.to_date(F.col('date')).alias('holiday_date'),
        F.col('holiday_name')
    )
    .dropDuplicates(['holiday_date'])
)

# Left join por data
df_with_holidays = (
    df_with_weather
    .join(
        F.broadcast(df_holidays),
        df_with_weather['date'] == df_holidays['holiday_date'],
        how='left'
    )
    .drop('holiday_date')
    .withColumn('is_holiday',
        F.col('holiday_name').isNotNull().cast(BooleanType())
    )
)

n_holiday_trips = df_with_holidays.filter(F.col('is_holiday')).count()
print(f"Viagens em feriados: {n_holiday_trips:,}")
print("\nTop feriados por volume de viagens:")
(
    df_with_holidays
    .filter(F.col('is_holiday'))
    .groupBy('holiday_name')
    .count()
    .orderBy(F.desc('count'))
    .show(10, truncate=False)
)

Viagens em feriados: 104,274

Top feriados por volume de viagens:
+--------------------------+-----+
|holiday_name              |count|
+--------------------------+-----+
|Martin Luther King Jr. Day|50947|
|New Year's Day            |27877|
|Christmas Day             |25450|
+--------------------------+-----+



In [17]:
# ============================================================
# 3.7 Join com Silver Stations (origem e destino)
# ============================================================

silver_stations_path = str(SILVER_STATIONS).replace('\\', '/')

df_stations_silver = spark.read.format('delta').load(silver_stations_path)

# Join para estação de início
df_station_start = df_stations_silver.select(
    F.col('station_id').alias('_start_id'),
    F.col('station_name').alias('start_station_name_clean'),
    F.col('capacity').alias('start_station_capacity'),
    F.col('nyc_zone').alias('start_zone')
)

# Join para estação de fim
df_station_end = df_stations_silver.select(
    F.col('station_id').alias('_end_id'),
    F.col('station_name').alias('end_station_name_clean'),
    F.col('capacity').alias('end_station_capacity'),
    F.col('nyc_zone').alias('end_zone')
)

df_with_stations = (
    df_with_holidays
    # Join estação início
    .join(
        F.broadcast(df_station_start),
        df_with_holidays['start_station_id'] == df_station_start['_start_id'],
        how='left'
    )
    .drop('_start_id')
    # Join estação fim
    .join(
        F.broadcast(df_station_end),
        df_with_holidays['end_station_id'] == df_station_end['_end_id'],
        how='left'
    )
    .drop('_end_id')
)

print(f"Registros após joins de estações: {df_with_stations.count():,}")
print("\nDistribuição por zona de origem:")
df_with_stations.groupBy('start_zone').count().orderBy(F.desc('count')).show()

Registros após joins de estações: 3,891,292

Distribuição por zona de origem:
+--------------------+-------+
|          start_zone|  count|
+--------------------+-------+
|           Manhattan|2999034|
|            Brooklyn| 742924|
|              Queens|  69020|
|               Bronx|  55488|
|                NULL|  24710|
|Jersey City / Hob...|    116|
+--------------------+-------+



In [18]:
# ============================================================
# 3.8 Schema Final e Escrita como Delta Lake
# ============================================================

# Schema Silver Trips — colunas ordenadas semanticamente
df_silver_trips = df_with_stations.select(

    # --- Identificador ---
    'ride_id',

    # --- Tipo e usuário ---
    'rideable_type',
    'member_casual',

    # --- Timestamps ---
    'started_at',
    'ended_at',

    # --- Métricas de viagem ---
    'trip_duration_sec',
    'distance_km',
    'avg_speed_kmh',

    # --- Features temporais ---
    'date',
    'year',
    'month',
    'hour_of_day',
    'day_of_week',
    'day_name',
    'is_weekend',
    'season',

    # --- Feriados ---
    'is_holiday',
    'holiday_name',

    # --- Clima no momento da viagem ---
    'weather_temp_c',
    'weather_precip_mm',
    'weather_is_raining',
    'weather_code',
    'weather_category',

    # --- Estação de origem ---
    'start_station_id',
    F.coalesce(F.col('start_station_name_clean'), F.col('start_station_name')).alias('start_station_name'),
    'start_lat',
    'start_lng',
    'start_station_capacity',
    'start_zone',

    # --- Estação de destino ---
    'end_station_id',
    F.coalesce(F.col('end_station_name_clean'), F.col('end_station_name')).alias('end_station_name'),
    'end_lat',
    'end_lng',
    'end_station_capacity',
    'end_zone',

    # --- Metadados ---
    F.current_timestamp().alias('silver_ingestion_ts')
)

silver_trips_path = str(SILVER_TRIPS).replace('\\', '/')

(
    df_silver_trips
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('year', 'month')    # partição eficiente para consultas por período
    .save(silver_trips_path)
)

dt_trips = DeltaTable.forPath(spark, silver_trips_path)
dt_trips.history(1).select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

final_count = spark.read.format('delta').load(silver_trips_path).count()
print(f"\nSilver Trips gravado: {final_count:,} registros")
print(f"Path: {silver_trips_path}")
print(f"\nTaxa de retenção: {final_count/total_raw*100:.1f}% dos dados Bronze")

+-------+-----------------------+---------+-----------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                       |
+-------+-----------------------+---------+-----------------------------------------------------------------------+
|0      |2026-05-04 16:16:11.101|WRITE    |{numFiles -> 18, numOutputRows -> 3891292, numOutputBytes -> 228506436}|
+-------+-----------------------+---------+-----------------------------------------------------------------------+


Silver Trips gravado: 3,891,292 registros
Path: /home/jovyan/work/dados/silver/trips

Taxa de retenção: 99.6% dos dados Bronze


---
## 4. Validação e Data Quality

Checks aplicados para garantir a qualidade da camada Silver:
- Contagem Bronze vs Silver (taxa de retenção)
- Análise de nulos por coluna
- Distribuição das features calculadas
- Delta Lake History (audit trail)
- Exemplo de Time Travel


In [19]:
# ============================================================
# 4.1 Análise de Nulos — Silver Trips
# ============================================================

df_silver = spark.read.format('delta').load(str(SILVER_TRIPS).replace('\\', '/'))

print(f"Total Silver Trips: {df_silver.count():,} registros")

print("\n--- Contagem de Nulos por Coluna ---")

null_counts = []
for col_name in df_silver.columns:
    n = df_silver.filter(F.col(col_name).isNull()).count()
    null_counts.append((col_name, n, f"{n/df_silver.count()*100:.2f}%"))

null_df = spark.createDataFrame(null_counts, ['coluna', 'nulos', 'pct'])
null_df.filter(F.col('nulos') > 0).orderBy(F.desc('nulos')).show(30, truncate=False)

Total Silver Trips: 3,891,292 registros

--- Contagem de Nulos por Coluna ---
+----------------------+-------+------+
|coluna                |nulos  |pct   |
+----------------------+-------+------+
|holiday_name          |3787018|97.32%|
|end_station_capacity  |60049  |1.54% |
|end_zone              |60049  |1.54% |
|start_station_capacity|24710  |0.64% |
|start_zone            |24710  |0.64% |
|end_station_id        |7      |0.00% |
|end_station_name      |7      |0.00% |
+----------------------+-------+------+



In [20]:
# ============================================================
# 4.2 Distribuições e Estatísticas Descritivas
# ============================================================

print("=== Estatísticas: Duração e Distância ===")
df_silver.select(
    'trip_duration_sec', 'distance_km', 'avg_speed_kmh',
    'weather_temp_c', 'weather_precip_mm'
).describe().show()

print("\n=== Trips por Estação do Ano ===")
df_silver.groupBy('season').count().orderBy('season').show()

print("\n=== Trips por Condição Climática ===")
df_silver.groupBy('weather_category').count().orderBy(F.desc('count')).show()

print("\n=== Trips em Feriados vs Dias Normais ===")
df_silver.groupBy('is_holiday', 'is_weekend').count().orderBy('is_holiday', 'is_weekend').show()

print("\n=== Top 10 Zonas de Origem ===")
df_silver.groupBy('start_zone').count().orderBy(F.desc('count')).show()

=== Estatísticas: Duração e Distância ===
+-------+------------------+------------------+------------------+------------------+--------------------+
|summary| trip_duration_sec|       distance_km|     avg_speed_kmh|    weather_temp_c|   weather_precip_mm|
+-------+------------------+------------------+------------------+------------------+--------------------+
|  count|           3891292|           3891292|           3891292|           3891292|             3891292|
|   mean| 612.1826434510697|1.7338612987665294|11.229676626683352|0.5967330130969106|0.047430082347973586|
| stddev|1013.0073624980374| 1.442747700847486|3.8630660232105263| 4.774601571625286|   0.268324766005568|
|    min|                61|               0.0|               0.0|             -15.6|                 0.0|
|    max|             86385|            26.462|             46.55|              14.1|                 4.4|
+-------+------------------+------------------+------------------+------------------+-----------------

In [21]:
# ============================================================
# 4.3 Delta Lake — Histórico e Time Travel
# ============================================================

from delta.tables import DeltaTable

print("=== Histórico de versões — Silver Trips ===")
dt_trips = DeltaTable.forPath(spark, str(SILVER_TRIPS).replace('\\', '/'))
dt_trips.history().select(
    'version', 'timestamp', 'operation', 'operationMetrics'
).show(truncate=False)

print("\n=== Time Travel — consulta na versão 0 ===")
df_v0 = (
    spark.read
    .format('delta')
    .option('versionAsOf', 0)
    .load(str(SILVER_TRIPS).replace('\\', '/'))
)
print(f"Registros na versão 0: {df_v0.count():,}")

print("\n=== Histórico de versões — Silver Weather ===")
dt_weather = DeltaTable.forPath(spark, str(SILVER_WEATHER).replace('\\', '/'))
dt_weather.history().select('version', 'timestamp', 'operation').show(truncate=False)

print("\n=== Histórico de versões — Silver Stations ===")
dt_stations = DeltaTable.forPath(spark, str(SILVER_STATIONS).replace('\\', '/'))
dt_stations.history().select('version', 'timestamp', 'operation').show(truncate=False)

=== Histórico de versões — Silver Trips ===
+-------+-----------------------+---------+-----------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                       |
+-------+-----------------------+---------+-----------------------------------------------------------------------+
|0      |2026-05-04 16:16:11.101|WRITE    |{numFiles -> 18, numOutputRows -> 3891292, numOutputBytes -> 228506436}|
+-------+-----------------------+---------+-----------------------------------------------------------------------+


=== Time Travel — consulta na versão 0 ===
Registros na versão 0: 3,891,292

=== Histórico de versões — Silver Weather ===
+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-05-04 15:55:04.122|WRITE    |
+-------+-----------------------+---------+


=== Histórico de versões — Si

In [22]:
# ============================================================
# 4.4 Registrar Tabelas Delta no Catálogo Spark SQL
# ============================================================

# Registrar como views temporárias para Spark SQL
spark.read.format('delta').load(str(SILVER_TRIPS).replace('\\', '/')).createOrReplaceTempView('silver_trips')
spark.read.format('delta').load(str(SILVER_WEATHER).replace('\\', '/')).createOrReplaceTempView('silver_weather')
spark.read.format('delta').load(str(SILVER_STATIONS).replace('\\', '/')).createOrReplaceTempView('silver_stations')

print("=== Query SQL — Top 10 estações de origem por volume ===")
spark.sql("""
    SELECT
        start_station_name,
        start_zone,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_duration_sec)/60, 1) AS avg_duration_min,
        ROUND(AVG(distance_km), 2) AS avg_distance_km,
        SUM(CAST(is_holiday AS INT)) AS holiday_trips
    FROM silver_trips
    GROUP BY start_station_name, start_zone
    ORDER BY total_trips DESC
    LIMIT 10
""").show(truncate=False)

print("\n=== Query SQL — Demanda por hora e clima ===")
spark.sql("""
    SELECT
        hour_of_day,
        weather_category,
        COUNT(*) AS total_trips,
        ROUND(AVG(trip_duration_sec)/60, 1) AS avg_min,
        ROUND(AVG(distance_km), 2) AS avg_km
    FROM silver_trips
    WHERE weather_category IS NOT NULL
    GROUP BY hour_of_day, weather_category
    ORDER BY hour_of_day, total_trips DESC
""").show(50, truncate=False)

=== Query SQL — Top 10 estações de origem por volume ===
+------------------------+----------+-----------+----------------+---------------+-------------+
|start_station_name      |start_zone|total_trips|avg_duration_min|avg_distance_km|holiday_trips|
+------------------------+----------+-----------+----------------+---------------+-------------+
|W 21 St & 6 Ave         |Manhattan |15894      |8.9             |1.37           |376          |
|W 31 St & 7 Ave         |Manhattan |13537      |9.6             |1.61           |334          |
|Lafayette St & E 8 St   |Manhattan |13398      |8.7             |1.36           |347          |
|Pier 61 At Chelsea Piers|Manhattan |13174      |10.7            |2.05           |328          |
|Broadway & E 14 St      |Manhattan |12715      |9.1             |1.41           |303          |
|9 Ave & W 33 St         |Manhattan |12686      |10.0            |1.76           |189          |
|11 Ave & W 41 St        |Manhattan |12103      |9.6             |1.69

In [23]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada!")

SparkSession encerrada!


---
## 5. Resumo da Camada Silver

| Tabela Delta | Conteúdo | Partição | Features |
|---|---|---|---|
| `silver/trips/` | Viagens limpas e enriquecidas | `year, month` | Haversine, clima, feriados, zonas NYC |
| `silver/weather/` | Clima horário limpo | `year, month` | Chave de join, flag chuva |
| `silver/stations/` | Estações deduplicadas | Sem partição | Zona NYC, capacidade |

**Garantias da camada Silver:**
- Zero duplicatas por `ride_id`
- Durações validas: 60s a 86400s
- Coordenadas dentro da area metropolitana de NYC
- Schema enforced pelo Delta Lake
- ACID transactions — sem leituras sujas
- Time travel disponível por versão ou timestamp

**Próximos passos → Gold Layer (Notebook 04):**
- `gold/agg_demand_hourly/` — demanda agregada por estação × hora (para modelo preditivo)
- `gold/agg_station_balance/` — fluxo líquido por estação por dia (rebalanceamento)
- `gold/agg_weather_impact/` — correlação chuva/temperatura × demanda
- `gold/fact_trips/` — tabela fato otimizada com ZORDER para BI
- `gold/dim_stations/` e `gold/dim_time/` — dimensões do modelo estrela
